# Encode representative sequences with legacy NTv3
Use `InstaDeepAI/NTv3_650M_pre`, not the revised `NTv3_pretrained`. Install `.[ntv3,notebook]`, accept the legacy model and code repository terms, and authenticate with Hugging Face. See [the sequence guide](../docs/SEQUENCES.md).

This notebook does not download or load weights unless `RUN_ENCODING` is set to `True`. FASTA IDs must match the abundance table's feature IDs.

In [ ]:
from pathlib import Path
from itertools import islice
from mgm2.sequences import encode_fasta, iter_fasta, LEGACY_MODEL_ID, LEGACY_MODEL_REVISION

FASTA = Path('representative_sequences.fasta')
OUTPUT = Path('outputs/asv_ntv3.h5')
LOCAL_MODEL = None  # Or Path('/path/to/NTv3_650M_pre')
DEVICE = 'cpu'  # Set to 'cuda:0' for GPU encoding
RUN_ENCODING = False
print(LEGACY_MODEL_ID, LEGACY_MODEL_REVISION)

In [ ]:
if FASTA.exists():
    for name, sequence in islice(iter_fasta(FASTA), 5):
        print(name, len(sequence), 'bases')
else:
    print('Set FASTA to your representative DNA sequence file.')

In [ ]:
if RUN_ENCODING:
    encode_fasta(FASTA, OUTPUT, model_path=LOCAL_MODEL, device=DEVICE,
                 batch_size=1, trust_remote_code=True)
else:
    print('Encoding is disabled. Configure your input and device, then set RUN_ENCODING=True.')

In [ ]:
import json
import h5py
from mgm2 import EmbeddingBank

if OUTPUT.exists():
    with h5py.File(OUTPUT) as bank:
        print(json.dumps(json.loads(bank.attrs['provenance_json']), indent=2))
    with EmbeddingBank(OUTPUT) as bank:
        vectors, sources = bank.resolve(bank.ids[:3])
        print(vectors.shape, sources)

## Continue with community inference
Pass the generated bank to the existing CLI or the Python table interface.
```bash
mgm2-embed --model "$MODEL_ROOT/models/small" --table communities.csv \
  --embeddings outputs/asv_ntv3.h5 --output outputs/communities.npz --device cpu
```
Sequence encoding and taxonomy fallback are independent routes. See [01_getting_started.ipynb](01_getting_started.ipynb) for community representations and SAE features.